## Packages importing 

In [ ]:
!pip install matplotlib seaborn pandas numpy scikit-learn datasist imbalanced-learn category-encoders xgboost joblib

In [ ]:
# Packages for EDA 
import matplotlib.pyplot as plt
import seaborn as sns 
import pandas as pd 
import numpy as np 

# Data Preprocessing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import PowerTransformer
from datasist.structdata import detect_outliers
from sklearn.metrics import mean_squared_error
from imblearn.over_sampling import SMOTE
from sklearn.impute import SimpleImputer
import category_encoders as ce
import re 

# Modeling and evaluation 
from sklearn.experimental import enable_hist_gradient_boosting
from sklearn.ensemble import (
    BaggingClassifier,
    ExtraTreesClassifier,
    RandomForestClassifier,
    StackingClassifier,
    HistGradientBoostingClassifier
)
from xgboost import XGBClassifier
from sklearn.metrics import classification_report 
import joblib

# Packages options 
sns.set(rc={'figure.figsize': [14, 7]}, font_scale=1.2) # Standard figure size for all 
np.seterr(divide='ignore', invalid='ignore', over='ignore') ;

import warnings 
warnings.filterwarnings("ignore")

## Reading Data 

In [ ]:
df = pd.read_csv("train.csv",low_memory=False)

In [ ]:
df.sample(10)

### Issues with this dataset: 
1. The following headings are supposed to be numerical but appear as categorical datatypes: Age, Annual_Income, Num_of_Loan, Num_of_Delayed_Payment, Changed_Credit_Limit, Amount_invested_monthly, Outstanding_Debt Credit_Mix, Monthly_Balance
2. Remove these headings: ID, Name and SSN (Not useful)
3. Remove missing data
4. Credit_Mix has value a value "-" which needs to be removed/fixed
5. Num_Credit_Card has __zeros__
6. Type_of_Loan nees to be rewritten
7. Negative values exist in the header: Num_Bank_Accounts
8. Outliers need to be removed
9. Missing data need to be filled/removed
10. Target column does not have even balance of outputs
11. Following headers need more fixing to balance out dataset: Credit_History_Age,Payment_of_Min_Amount,Payment_Behaviour,'Credit_Mix'

## Data cleaning

In [ ]:
del df['ID'] # Identification 
del df['Name'] # Name of client 
del df['SSN'] # SSN (social security number of a person)

#### Fixing Numerical Columns to be of float data-type and removing "-" columns

1. replace _  
2. convert into float 

In [ ]:
to_be_fixed = ['Age', 'Annual_Income', 'Num_of_Loan', 'Num_of_Delayed_Payment', 'Changed_Credit_Limit', 'Amount_invested_monthly', 'Outstanding_Debt' , 'Monthly_Balance']

In [ ]:
def fix_nums(num):
    try : 
        return float(num.replace("_",""))
    except :
        return np.nan

In [ ]:
for col in to_be_fixed :
    df[col] = df[col].apply(fix_nums)

### Transform Type_of_Loan header

In [ ]:
## transform the Type_of_Loan column, which contains categorical data, into multiple binary columns.
for i in df['Type_of_Loan'].value_counts().head(9).index[1:] : 
    df[i] = df['Type_of_Loan'].str.contains(i)

del df['Type_of_Loan']

#### Make Num_Bank_Accounts to be positive always

In [ ]:
# number of bank accounts must be positive
df['Num_Bank_Accounts'] = df['Num_Bank_Accounts'].apply(lambda x :abs (x))

#### Non-Zero for Num_Credit_Card

In [ ]:
# number of credit cards must be atleast 1
df['Num_Credit_Card'].replace(0,1,inplace=True)

#### Convert credit history ages under Credit_History_Age to all integers

In [ ]:
def history_age(age):
    try : 
        years = int("".join(re.findall('[0-9]',''.join(age.split("and")[0]))))
        month = int("".join(re.findall('[0-9]',''.join(age.split("and")[1]))))
        return years*12 + month
    except :
        return np.nan

In [ ]:
df['Credit_History_Age'] = df['Credit_History_Age'].apply(history_age)

#### Payment_of_Min_Amount

In [ ]:
df['Payment_of_Min_Amount'].replace("NM","No",inplace=True)

In [ ]:
df['Payment_of_Min_Amount'].value_counts()

#### Payment_Behaviour

In [ ]:
df['Payment_Behaviour']= df['Payment_Behaviour'].replace("!@9#%8",'Medium_spent_Medium_value_payments')

In [ ]:
df['Payment_Behaviour'].value_counts()

### Occupation

In [ ]:
df['Occupation'].value_counts()

In [ ]:
occs = df['Occupation'].value_counts().index[1:]
occs

In [ ]:
# for the specified customer, identify the most common occupation in their records
# replace missing values of "_______" with the most common occupation for that customer
id_ = "CUS_0xb891"
oc = df[df['Customer_ID'] == id_]['Occupation'].mode()[0]
df[df['Customer_ID'] == id_].replace("_______",oc)

In [ ]:
# for _id_ in df[df['Occupation'] == "_______"]['Customer_ID'] : 
#     oc = df[df['Customer_ID'] == _id_]['Occupation'].mode()[0]
#     df[df['Customer_ID'] == _id_] = df[df['Customer_ID'] == _id_].replace("_______",oc)

In [ ]:
missing_occupation = df['Occupation'] == '_______'
# Find the most frequent occupation for each Customer_ID
most_frequent_occupation = df[~missing_occupation].groupby('Customer_ID')['Occupation'].agg(lambda x: x.mode()[0])
# Replace missing Occupation values
df.loc[missing_occupation, 'Occupation'] = df.loc[missing_occupation, 'Customer_ID'].map(most_frequent_occupation)


In [ ]:
df['Occupation'].value_counts()

In [ ]:
df['Occupation'] = df['Occupation'].replace("_______",df['Occupation'].mode()[0])

In [ ]:
df['Occupation'].value_counts()

In [ ]:
df['Credit_Mix'].value_counts()

In [ ]:
m = {
    "Bad":0,
    "Standard":1,
    "Good":2,
    "_":np.nan
}

In [ ]:
df['Credit_Mix'] = df['Credit_Mix'].map(m)

In [ ]:
df['Credit_Mix'].value_counts()

### Advanced Handling Missing Data 

In [ ]:
# edit Columns from bool to int 
for col in list(df.columns[-8:]):
    df[col] = df[col].astype(float)

In [ ]:
# IDs = 1 
# for ID in df['Customer_ID'].unique() :
#     df['Customer_ID'] = df['Customer_ID'].replace(ID,IDs)
#     IDs += 1 

In [ ]:
# Use pd.factorize to assign unique integers to each unique Customer_ID
df['Customer_ID'] = pd.factorize(df['Customer_ID'])[0] + 1

In [ ]:
from sklearn.impute import KNNImputer
imputer = KNNImputer(n_neighbors=1)

In [ ]:
Numericals = df.select_dtypes(exclude='object').columns[1:]
Numericals

In [ ]:
# Replace infinity values with NaN
df.replace([np.inf, -np.inf], np.nan, inplace=True)

# Check for NaN values and handle them (e.g., by replacing them with the mean of the column)
for col in Numericals:
    if df[col].isnull().any():
        df[col].fillna(df[col].mean(), inplace=True)

In [ ]:
from sklearn.impute import SimpleImputer

# Initialize the imputer
imputer = SimpleImputer(strategy='mean')

# Apply the imputer to each numerical column
for col in Numericals:
    df[[col]] = imputer.fit_transform(df[[col]])


In [ ]:
for col in Numericals[1:]:
    imputer.fit(df[['Customer_ID',col]])
    df[['Customer_ID',col]] = imputer.transform(df[['Customer_ID',col]])

In [ ]:
df.info()

In [ ]:
imputer = SimpleImputer(strategy="most_frequent")
imputer.fit(df[['Payment_Behaviour']])
df[['Payment_Behaviour']] = imputer.transform(df[['Payment_Behaviour']])

### Handling Outliers 

In [ ]:
## replace Outliers with median 
for col in Numericals :
    outliers_indecies = detect_outliers(df,0,[col])
    median = df[col].median()
    df[col].iloc[outliers_indecies] = median

In [ ]:
df.to_csv("processed2.csv")

# Data Preprocessing

### Handling Catogerical 

In [ ]:
df.select_dtypes(include="object")

In [ ]:
df['Month'].value_counts()

In [ ]:
df['Credit_Score'].value_counts()

In [ ]:
m = {
    "Poor":0,
    "Standard":1,
    "Good":2
}

In [ ]:
df['Credit_Score'] = df['Credit_Score'].map(m)

In [ ]:
df['Credit_Score'].value_counts()

In [ ]:
del df['Customer_ID']

In [ ]:
df = pd.get_dummies(df,drop_first=True)

In [ ]:
df.info()

In [ ]:
df.to_csv("final_processed_data.csv")

In [ ]:
# df = pd.read_csv("final_processed_data.csv",low_memory=False)
# df = df.iloc[:, 1:]

In [ ]:
df.shape

### Data Spliting 
- Try Resampling 

In [ ]:
# define dataset
X, y = df.drop("Credit_Score",axis=1).values , df["Credit_Score"] 

In [ ]:
X_shape = X.shape
y_shape = y.shape

X_shape, y_shape

## Apply oversampling
- As our data is not fully balanced, we intend to use oversampling to balance it out as much as possible using SMOTE
- SMOTE (Synthetic Minority Over-sampling Technique) is a popular method used to address class imbalance in datasets, particularly in the context of binary classification problems.

In [ ]:
y.value_counts(normalize=True)

In [ ]:
from imblearn.over_sampling import SMOTE
rus = SMOTE(sampling_strategy='auto')
X_data_rus, y_data_rus = rus.fit_resample(X, y)

In [ ]:
y_data_rus.value_counts(normalize=True)

In [ ]:
# split into train and test sets
X_train, X_test, y_train, y_test = train_test_split(X_data_rus, y_data_rus, test_size=0.3, random_state=42,stratify=y_data_rus)

### Handling Numerical 
- Using Power transformer to avoid Data Skewness
- The PowerTransformer is a tool in scikit-learn used to apply power transformations to numerical data to stabilize variance and make the data more Gaussian-like (i.e., normally distributed)

In [ ]:
scalar = PowerTransformer(method='yeo-johnson', standardize=True).fit(X_train)

In [ ]:
X_train = scalar.transform(X_train)
X_test = scalar.transform(X_test)

In [ ]:
print("Number of features used for fitting the scaler:", scalar.n_features_in_)


# Modeling and Evaluation


#### Model Building

In [ ]:
bagging = BaggingClassifier(n_jobs=-1)
extraTrees = ExtraTreesClassifier(max_depth=10, n_jobs=-1)
randomForest = RandomForestClassifier(n_jobs=-1)
histGradientBoosting = HistGradientBoostingClassifier()
XGB = XGBClassifier(n_jobs=-1)

model = StackingClassifier([
    ('bagging', bagging),
    ('extraTress', extraTrees),
    ('randomforest', randomForest),
    ('histGradientBoosting', histGradientBoosting),
    ('XGB', XGB)
], n_jobs=-1)


#### Model fitting

In [ ]:
model.fit(X_train, y_train)

#### model evaluation

In [ ]:
print("Train Score: ",model.score(X_train, y_train))

In [ ]:
print("Test Score: ",model.score(X_test, y_test))

In [ ]:
y_pred = model.predict(X_test)

In [ ]:
print(classification_report(y_pred,y_test))

#### Saving the model

In [ ]:
import joblib

# Save the StackingClassifier model
joblib.dump(model, 'stacking_model_2.pkl')

# Save the scaler
joblib.dump(scalar, 'stacking_model_scaler_2.pkl')

### Test with examples ###

In [ ]:
import pandas as pd
import joblib
case_1 = {
    'Age': 45,
    'Annual_Income': 120000,
    'Monthly_Inhand_Salary': 10000,
    'Num_Bank_Accounts': 5,
    'Num_Credit_Card': 8,
    'Interest_Rate': 3,
    'Num_of_Loan': 2,
    'Delay_from_due_date': 0,
    'Num_of_Delayed_Payment': 1,
    'Changed_Credit_Limit': 5000,
    'Num_Credit_Inquiries': 2,
    'Credit_Mix': 1,
    'Outstanding_Debt': 10000,
    'Credit_Utilization_Ratio': 20,
    'Credit_History_Age': 20,
    'Total_EMI_per_month': 1500,
    'Amount_invested_monthly': 2000,
    'Monthly_Balance': 5000,
    'Credit-Builder Loan': 0.0,
    'Personal Loan': 1.0,
    'Debt Consolidation Loan': 0.0,
    'Student Loan': 0.0,
    'Payday Loan': 0.0,
    'Mortgage Loan': 1.0,
    'Auto Loan': 1.0,
    'Home Equity Loan': 0.0,
    'Month_August': 0,
    'Month_December': 0,
    'Month_February': 1,
    'Month_January': 0,
    'Month_July': 0,
    'Month_June': 0,
    'Month_March': 0,
    'Month_May': 0,
    'Month_November': 0,
    'Month_October': 0,
    'Month_September': 0,
    'Occupation_Architect': 0,
    'Occupation_Developer': 0,
    'Occupation_Doctor': 0,
    'Occupation_Engineer': 1,
    'Occupation_Entrepreneur': 0,
    'Occupation_Journalist': 0,
    'Occupation_Lawyer': 0,
    'Occupation_Manager': 1,
    'Occupation_Mechanic': 0,
    'Occupation_Media_Manager': 0,
    'Occupation_Musician': 0,
    'Occupation_Others': 0,
    'Occupation_Scientist': 0,
    'Occupation_Teacher': 0,
    'Occupation_Writer': 0,
    'Payment_of_Min_Amount_Yes': 1,
    'Payment_Behaviour_High_spent_Medium_value_payments': 1,
    'Payment_Behaviour_High_spent_Small_value_payments': 0,
    'Payment_Behaviour_Low_spent_Large_value_payments': 0,
    'Payment_Behaviour_Low_spent_Medium_value_payments': 0,
    'Payment_Behaviour_Low_spent_Small_value_payments': 0,
    'Payment_Behaviour_Medium_spent_Medium_value_payments': 0
}

case_poor = {
    'Age': 25,
    'Annual_Income': 20000,
    'Monthly_Inhand_Salary': 1500,
    'Num_Bank_Accounts': 2,
    'Num_Credit_Card': 1,
    'Interest_Rate': 20,
    'Num_of_Loan': 4,
    'Delay_from_due_date': 30,
    'Num_of_Delayed_Payment': 5,
    'Changed_Credit_Limit': 200,
    'Num_Credit_Inquiries': 10,
    'Credit_Mix': 0,
    'Outstanding_Debt': 50000,
    'Credit_Utilization_Ratio': 90,
    'Credit_History_Age': 1,
    'Total_EMI_per_month': 3000,
    'Amount_invested_monthly': 100,
    'Monthly_Balance': -500,
    'Credit-Builder Loan': 0.0,
    'Personal Loan': 1.0,
    'Debt Consolidation Loan': 0.0,
    'Student Loan': 1.0,
    'Payday Loan': 1.0,
    'Mortgage Loan': 0.0,
    'Auto Loan': 0.0,
    'Home Equity Loan': 0.0,
    'Month_August': 0,
    'Month_December': 0,
    'Month_February': 0,
    'Month_January': 1,
    'Month_July': 0,
    'Month_June': 0,
    'Month_March': 0,
    'Month_May': 0,
    'Month_November': 0,
    'Month_October': 0,
    'Month_September': 0,
    'Occupation_Architect': 0,
    'Occupation_Developer': 0,
    'Occupation_Doctor': 0,
    'Occupation_Engineer': 0,
    'Occupation_Entrepreneur': 0,
    'Occupation_Journalist': 0,
    'Occupation_Lawyer': 0,
    'Occupation_Manager': 0,
    'Occupation_Mechanic': 1,
    'Occupation_Media_Manager': 0,
    'Occupation_Musician': 0,
    'Occupation_Others': 0,
    'Occupation_Scientist': 0,
    'Occupation_Teacher': 0,
    'Occupation_Writer': 0,
    'Payment_of_Min_Amount_Yes': 0,
    'Payment_Behaviour_High_spent_Medium_value_payments': 0,
    'Payment_Behaviour_High_spent_Small_value_payments': 1,
    'Payment_Behaviour_Low_spent_Large_value_payments': 0,
    'Payment_Behaviour_Low_spent_Medium_value_payments': 0,
    'Payment_Behaviour_Low_spent_Small_value_payments': 0,
    'Payment_Behaviour_Medium_spent_Medium_value_payments': 0
}


# Convert the case to DataFrame
case_df = pd.DataFrame([case_poor])

# Ensure all expected columns are present
expected_features = [
    'Age', 'Annual_Income', 'Monthly_Inhand_Salary', 'Num_Bank_Accounts',
    'Num_Credit_Card', 'Interest_Rate', 'Num_of_Loan', 'Delay_from_due_date',
    'Num_of_Delayed_Payment', 'Changed_Credit_Limit', 'Num_Credit_Inquiries',
    'Credit_Mix', 'Outstanding_Debt', 'Credit_Utilization_Ratio',
    'Credit_History_Age', 'Total_EMI_per_month', 'Amount_invested_monthly',
    'Monthly_Balance', 'Credit-Builder Loan', 'Personal Loan',
    'Debt Consolidation Loan', 'Student Loan', 'Payday Loan', 'Mortgage Loan',
    'Auto Loan', 'Home Equity Loan', 'Month_August', 'Month_December',
    'Month_February', 'Month_January', 'Month_July', 'Month_June', 'Month_March',
    'Month_May', 'Month_November', 'Month_October', 'Month_September',
    'Occupation_Architect', 'Occupation_Developer', 'Occupation_Doctor',
    'Occupation_Engineer', 'Occupation_Entrepreneur', 'Occupation_Journalist',
    'Occupation_Lawyer', 'Occupation_Manager', 'Occupation_Mechanic',
    'Occupation_Media_Manager', 'Occupation_Musician', 'Occupation_Others',
    'Occupation_Scientist', 'Occupation_Teacher', 'Occupation_Writer',
    'Payment_of_Min_Amount_Yes',
    'Payment_Behaviour_High_spent_Medium_value_payments',
    'Payment_Behaviour_High_spent_Small_value_payments',
    'Payment_Behaviour_Low_spent_Large_value_payments',
    'Payment_Behaviour_Low_spent_Medium_value_payments',
    'Payment_Behaviour_Low_spent_Small_value_payments',
    'Payment_Behaviour_Medium_spent_Medium_value_payments'
]

# Ensure the columns are in the correct order
case_df = case_df[expected_features]

scalar = joblib.load('stacking_model_scaler_2.pkl')
model = joblib.load('stacking_model_2.pkl')
# Transform the input data using the saved scaler
case_transformed = scalar.transform(case_df)

# Make a prediction
prediction = model.predict(case_transformed)
m = {
    0: "Poor",
    1: "Standard",
    2: "Good"
}
print(f"The model predicts: {m[prediction[0]]}")

In [ ]:
len(expected_features)

In [ ]:
# <class 'pandas.core.frame.DataFrame'>
# RangeIndex: 100000 entries, 0 to 99999
# Data columns (total 60 columns):
#  #   Column                                                Non-Null Count   Dtype  
# ---  ------                                                --------------   -----  
#  0   Age                                                   100000 non-null  float64
#  1   Annual_Income                                         100000 non-null  float64
#  2   Monthly_Inhand_Salary                                 100000 non-null  float64
#  3   Num_Bank_Accounts                                     100000 non-null  float64
#  4   Num_Credit_Card                                       100000 non-null  float64
#  5   Interest_Rate                                         100000 non-null  float64
#  6   Num_of_Loan                                           100000 non-null  float64
#  7   Delay_from_due_date                                   100000 non-null  float64
#  8   Num_of_Delayed_Payment                                100000 non-null  float64
#  9   Changed_Credit_Limit                                  100000 non-null  float64
#  10  Num_Credit_Inquiries                                  100000 non-null  float64
#  11  Credit_Mix                                            100000 non-null  float64
#  12  Outstanding_Debt                                      100000 non-null  float64
#  13  Credit_Utilization_Ratio                              100000 non-null  float64
#  14  Credit_History_Age                                    100000 non-null  float64
#  15  Total_EMI_per_month                                   100000 non-null  float64
#  16  Amount_invested_monthly                               100000 non-null  float64
#  17  Monthly_Balance                                       100000 non-null  float64
#  18  Credit_Score                                          100000 non-null  int64  
#  19  Credit-Builder Loan                                   100000 non-null  float64
#  20  Personal Loan                                         100000 non-null  float64
#  21  Debt Consolidation Loan                               100000 non-null  float64
#  22  Student Loan                                          100000 non-null  float64
#  23  Payday Loan                                           100000 non-null  float64
#  24  Mortgage Loan                                         100000 non-null  float64
#  25  Auto Loan                                             100000 non-null  float64
#  26  Home Equity Loan                                      100000 non-null  float64
#  27  Month_August                                          100000 non-null  bool   
#  28  Month_December                                        100000 non-null  bool   
#  29  Month_February                                        100000 non-null  bool   
#  30  Month_January                                         100000 non-null  bool   
#  31  Month_July                                            100000 non-null  bool   
#  32  Month_June                                            100000 non-null  bool   
#  33  Month_March                                           100000 non-null  bool   
#  34  Month_May                                             100000 non-null  bool   
#  35  Month_November                                        100000 non-null  bool   
#  36  Month_October                                         100000 non-null  bool   
#  37  Month_September                                       100000 non-null  bool   
#  38  Occupation_Architect                                  100000 non-null  bool   
#  39  Occupation_Developer                                  100000 non-null  bool   
#  40  Occupation_Doctor                                     100000 non-null  bool   
#  41  Occupation_Engineer                                   100000 non-null  bool   
#  42  Occupation_Entrepreneur                               100000 non-null  bool   
#  43  Occupation_Journalist                                 100000 non-null  bool   
#  44  Occupation_Lawyer                                     100000 non-null  bool   
#  45  Occupation_Manager                                    100000 non-null  bool   
#  46  Occupation_Mechanic                                   100000 non-null  bool   
#  47  Occupation_Media_Manager                              100000 non-null  bool   
#  48  Occupation_Musician                                   100000 non-null  bool   
#  49  Occupation_Others                                     100000 non-null  bool   
#  50  Occupation_Scientist                                  100000 non-null  bool   
#  51  Occupation_Teacher                                    100000 non-null  bool   
#  52  Occupation_Writer                                     100000 non-null  bool   
#  53  Payment_of_Min_Amount_Yes                             100000 non-null  bool   
#  54  Payment_Behaviour_High_spent_Medium_value_payments    100000 non-null  bool   
#  55  Payment_Behaviour_High_spent_Small_value_payments     100000 non-null  bool   
#  56  Payment_Behaviour_Low_spent_Large_value_payments      100000 non-null  bool   
#  57  Payment_Behaviour_Low_spent_Medium_value_payments     100000 non-null  bool   
#  58  Payment_Behaviour_Low_spent_Small_value_payments      100000 non-null  bool   
#  59  Payment_Behaviour_Medium_spent_Medium_value_payments  100000 non-null  bool   
# dtypes: bool(33), float64(26), int64(1)